In [1]:
import os

In [2]:
pwd("../")

'f:\\Condition2Cure\\notebook'

In [3]:
os.chdir("../")

In [4]:
pwd("../")

'f:\\Condition2Cure'

In [5]:
from pathlib import Path
from dataclasses import dataclass

In [6]:
from Condition2Cure.utils.helpers import *
from Condition2Cure.constants import *

In [7]:
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    features_path: Path
    model_path: Path
    max_iter: int
    test_size: float
    random_state: int


In [8]:

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])



    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_training
        params = self.params.model
        split = self.params.train_test_split

        create_directories([config.root_dir])

        model_training_config =  ModelTrainerConfig(
            root_dir=config.root_dir,
            features_path=config.features_path,
            model_path=config.model_path,
            max_iter=params.max_iter,
            test_size=split.test_size,
            random_state=split.random_state
        )

        return model_training_config


In [9]:
import os
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier
from Condition2Cure import logger

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        logger.info("Loading features...")
        data = np.load(self.config.features_path, allow_pickle=True).item()
        X, y = data["X"], data["y"]

        logger.info("Splitting train/test sets...")
        X_train, _, y_train, _ = train_test_split(
            X, y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y
        )

        logger.info("Training PassiveAggressiveClassifier ...")
        model = PassiveAggressiveClassifier(max_iter=self.config.max_iter)
        model.fit(X_train, y_train)

        os.makedirs(os.path.dirname(self.config.model_path), exist_ok=True)
        joblib.dump(model, self.config.model_path)
        logger.info(f"Model saved to {self.config.model_path}")


In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
    
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
except KeyError as e:
    logger.error(f"Missing key in configuration: {str(e)}")
except Exception as e:
    logger.error(f"Unexpected error: {str(e)}") 

[2025-05-21 12:07:56,666: INFO: helpers: yaml file: config\config.yaml loaded successfully]
[2025-05-21 12:07:56,673: INFO: helpers: yaml file: config\params.yaml loaded successfully]
[2025-05-21 12:07:56,679: INFO: helpers: yaml file: config\schema.yaml loaded successfully]
[2025-05-21 12:07:56,681: INFO: helpers: created directory at: artifacts]
[2025-05-21 12:07:56,683: INFO: helpers: created directory at: artifacts/model_training]
[2025-05-21 12:07:56,685: INFO: 809048413: Loading features...]
[2025-05-21 12:07:56,747: INFO: 809048413: Splitting train/test sets...]
[2025-05-21 12:07:56,972: INFO: 809048413: Training PassiveAggressiveClassifier ...]
[2025-05-21 12:07:59,622: INFO: 809048413: Model saved to artifacts/model_training/model.joblib]
